# PAD-ONAP Kaggle Notebook: Stacked LSTM Multi-Horizon Forecasting


This notebook is designed for Kaggle and CICDDoS2019-style CSV files. It is safe: it reads offline CSV data and does not generate packet traffic. Set PAD_ONAP_DATASET_DIR if auto-discovery under /kaggle/input is not enough.

Outputs are written under /kaggle/working so they can be downloaded as thesis evidence. Do not report any metric in the thesis unless it is produced by an executed notebook run and saved in the output artifacts.


This notebook replaces the Transformer-LSTM forecaster with a stacked LSTM. It keeps the ddos-train-v4 17-feature representation and predicts future traffic classes at +6w, +12w, +18w, and +24w.


In [ ]:
import os
import json
import time
import math
import glob
import pickle
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

DATASET_DIR = Path(os.environ.get("PAD_ONAP_DATASET_DIR", "/kaggle/input"))
if not DATASET_DIR.exists():
    DATASET_DIR = Path(".")

OUTPUT_DIR = Path("/kaggle/working/pad_onap_stacked_lstm_outputs")
MODELS_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
for d in [OUTPUT_DIR, MODELS_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CHUNKSIZE = int(os.environ.get("PAD_ONAP_CHUNKSIZE", "200000"))
WINDOW_SIZE = int(os.environ.get("PAD_ONAP_WINDOW_SIZE", "100"))
STEP = int(os.environ.get("PAD_ONAP_STEP", "50"))
BENIGN_STEP = int(os.environ.get("PAD_ONAP_BENIGN_STEP", "5"))
MAX_WINDOWS = int(os.environ.get("PAD_ONAP_MAX_WINDOWS", "120000"))
MIN_CLASS_WINDOWS = int(os.environ.get("PAD_ONAP_MIN_CLASS_WINDOWS", "10000"))

CLASS_NAMES = {
    0: "Normal",
    1: "UDP_Flood",
    2: "SYN_Flood",
    3: "HTTP_Flood",
    4: "ICMP_Flood",
    5: "Amplification",
    6: "Slow_rate",
}

FEATURE_NAMES = [
    "pkt_rate", "byte_rate", "src_ip_entropy", "dst_ip_entropy",
    "src_port_entropy", "dst_port_entropy", "proto_dist_tcp",
    "proto_dist_udp", "proto_dist_icmp", "syn_ratio", "fin_ratio",
    "avg_pkt_size", "pkt_size_std", "new_flows_rate",
    "flow_duration_mean", "inter_arrival_mean", "inter_arrival_std",
]

def normalize_col(name):
    return "".join(ch for ch in str(name).lower().strip() if ch.isalnum())

def col_lookup(df):
    return {normalize_col(c): c for c in df.columns}

def get_col(df, names, default=0.0):
    lookup = col_lookup(df)
    for name in names:
        key = normalize_col(name)
        if key in lookup:
            return pd.to_numeric(df[lookup[key]], errors="coerce").fillna(default).to_numpy()
    return np.full(len(df), default, dtype=np.float64)

def get_str_col(df, names):
    lookup = col_lookup(df)
    for name in names:
        key = normalize_col(name)
        if key in lookup:
            return df[lookup[key]].astype(str).fillna("").to_numpy()
    return np.array([""] * len(df), dtype=object)

def shannon_entropy(values):
    if len(values) == 0:
        return 0.0
    _, counts = np.unique(values, return_counts=True)
    probs = counts.astype(np.float64) / max(1, counts.sum())
    return float(-(probs * np.log2(probs + 1e-12)).sum())

def map_label(label):
    s = str(label).strip().lower()
    s_norm = s.replace("-", "_").replace(" ", "_")
    if s_norm in {"benign", "normal"}:
        return 0
    if "udp" in s_norm and "lag" not in s_norm:
        return 1
    if "udp_lag" in s_norm or "udplag" in s_norm:
        return 1
    if "syn" in s_norm:
        return 2
    if "webddos" in s_norm or "http" in s_norm:
        return 3
    if "icmp" in s_norm:
        return 4
    if any(x in s_norm for x in ["drdos", "dns", "ldap", "mssql", "netbios", "ntp", "snmp", "ssdp", "tftp", "portmap", "chargen", "amplification"]):
        return 5
    if any(x in s_norm for x in ["slowloris", "slowhttptest", "slow_rate", "slowrate"]):
        return 6
    return None

def find_label_column(df):
    lookup = col_lookup(df)
    for candidate in ["Label", "label", "Class", "class"]:
        key = normalize_col(candidate)
        if key in lookup:
            return lookup[key]
    return None

def extract_17_features(df_window):
    n = len(df_window)
    if n == 0:
        return np.zeros(len(FEATURE_NAMES), dtype=np.float32)

    dur = get_col(df_window, ["Flow Duration", "FlowDuration"], 1.0)
    dur = np.where(dur <= 0, 1.0, dur)
    dur_sum = max(float(np.sum(dur)), 1.0)

    fwd_pkts = get_col(df_window, ["Total Fwd Packets", "Tot Fwd Pkts", "TotalFwdPackets"], 0.0)
    bwd_pkts = get_col(df_window, ["Total Backward Packets", "Total Bwd packets", "Tot Bwd Pkts"], 0.0)
    total_pkts = fwd_pkts + bwd_pkts

    fwd_bytes = get_col(df_window, ["Total Length of Fwd Packets", "TotLen Fwd Pkts", "Total Length Fwd Packets"], 0.0)
    bwd_bytes = get_col(df_window, ["Total Length of Bwd Packets", "TotLen Bwd Pkts", "Total Length Bwd Packets"], 0.0)
    total_bytes = fwd_bytes + bwd_bytes

    pkt_rate = float(np.sum(total_pkts) / dur_sum * 1e6)
    byte_rate = float(np.sum(total_bytes) / dur_sum * 1e6)

    src_ip_entropy = shannon_entropy(get_str_col(df_window, ["Source IP", "Src IP", "SrcIP"]))
    dst_ip_entropy = shannon_entropy(get_str_col(df_window, ["Destination IP", "Dst IP", "DstIP"]))
    src_port_entropy = shannon_entropy(get_str_col(df_window, ["Source Port", "Src Port", "SrcPort"]))
    dst_port_entropy = shannon_entropy(get_str_col(df_window, ["Destination Port", "Dst Port", "DstPort"]))

    proto = get_col(df_window, ["Protocol"], 6).astype(int)
    proto_dist_tcp = float((proto == 6).sum() / max(n, 1))
    proto_dist_udp = float((proto == 17).sum() / max(n, 1))
    proto_dist_icmp = float((proto == 1).sum() / max(n, 1))

    syn_count = get_col(df_window, ["SYN Flag Count", "SYNFlagCount"], 0.0)
    fin_count = get_col(df_window, ["FIN Flag Count", "FINFlagCount"], 0.0)
    pkt_sum = max(float(np.sum(total_pkts)), 1.0)
    syn_ratio = float(np.sum(syn_count) / pkt_sum)
    fin_ratio = float(np.sum(fin_count) / pkt_sum)

    avg_pkt_size_raw = get_col(df_window, ["Average Packet Size", "Avg Packet Size"], np.nan)
    if np.isfinite(avg_pkt_size_raw).any():
        avg_pkt_size = float(np.nanmean(avg_pkt_size_raw))
    else:
        avg_pkt_size = float(np.sum(total_bytes) / pkt_sum)

    pkt_size_std_raw = get_col(df_window, ["Packet Length Std", "Pkt Len Std"], np.nan)
    if np.isfinite(pkt_size_std_raw).any():
        pkt_size_std = float(np.nanmean(pkt_size_std_raw))
    else:
        pkt_size_std = float(np.std(total_bytes / np.maximum(total_pkts, 1.0)))

    new_flows_rate = float(n / dur_sum * 1e6)
    flow_duration_mean = float(np.mean(dur))
    inter_arrival_mean = float(np.mean(get_col(df_window, ["Flow IAT Mean", "FlowIATMean"], 0.0)))
    inter_arrival_std = float(np.mean(get_col(df_window, ["Flow IAT Std", "FlowIATStd"], 0.0)))

    feats = np.array([
        pkt_rate, byte_rate, src_ip_entropy, dst_ip_entropy,
        src_port_entropy, dst_port_entropy, proto_dist_tcp,
        proto_dist_udp, proto_dist_icmp, syn_ratio, fin_ratio,
        avg_pkt_size, pkt_size_std, new_flows_rate,
        flow_duration_mean, inter_arrival_mean, inter_arrival_std,
    ], dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=1e9, neginf=-1e9)
    return feats

def audit_csv_files(csv_files):
    label_counts = defaultdict(int)
    total_rows = 0
    for f in csv_files:
        try:
            for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines="skip"):
                lcol = find_label_column(chunk)
                if lcol is None:
                    continue
                total_rows += len(chunk)
                mapped = chunk[lcol].map(map_label)
                for cls, cnt in mapped.dropna().astype(int).value_counts().items():
                    label_counts[int(cls)] += int(cnt)
        except Exception as exc:
            print(f"Audit skipped {f}: {exc}")
    return total_rows, dict(label_counts)

def build_windows(csv_files, max_windows=MAX_WINDOWS):
    X, y, segments = [], [], []
    class_windows = defaultdict(int)
    active = set(range(7))
    class_quota = {c: max(MIN_CLASS_WINDOWS, max_windows // max(1, len(active))) for c in active}
    class_quota[0] = max(class_quota[0], MIN_CLASS_WINDOWS)
    t0 = time.time()

    for file_idx, f in enumerate(csv_files):
        if len(X) >= max_windows:
            break
        file_count = 0
        carry = pd.DataFrame()
        try:
            for chunk in pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False, on_bad_lines="skip"):
                lcol = find_label_column(chunk)
                if lcol is None:
                    continue
                mapped = chunk[lcol].map(map_label)
                chunk = chunk.loc[mapped.notna()].copy()
                if len(chunk) == 0:
                    continue
                chunk["_class"] = mapped.loc[mapped.notna()].astype(int).to_numpy()
                df = pd.concat([carry, chunk], ignore_index=True)

                i = 0
                while i + WINDOW_SIZE <= len(df) and len(X) < max_windows:
                    w = df.iloc[i:i + WINDOW_SIZE]
                    counts = np.bincount(w["_class"].to_numpy(dtype=np.int64), minlength=7)
                    cls = int(np.argmax(counts))
                    step = BENIGN_STEP if cls == 0 else STEP
                    if class_windows[cls] < class_quota.get(cls, max_windows):
                        X.append(extract_17_features(w))
                        y.append(cls)
                        class_windows[cls] += 1
                        file_count += 1
                    i += step
                carry = df.iloc[i:].reset_index(drop=True)
        except Exception as exc:
            print(f"Window extraction skipped {f}: {exc}")
        if file_count > 0:
            segments.append(file_count)
        print(f"[{file_idx + 1}/{len(csv_files)}] {Path(f).name}: windows={file_count}, total={len(X)}, elapsed={time.time() - t0:.1f}s")

    if len(X) == 0:
        raise RuntimeError("No windows extracted. Check dataset path and label mapping.")

    return np.vstack(X).astype(np.float32), np.asarray(y, dtype=np.int64), segments

csv_files = sorted(glob.glob(str(DATASET_DIR / "**" / "*.csv"), recursive=True))
print(f"DATASET_DIR={DATASET_DIR}")
print(f"CSV files found: {len(csv_files)}")
for p in csv_files[:10]:
    print(" -", p)
if len(csv_files) > 10:
    print(f" ... {len(csv_files) - 10} more")

MAX_WINDOWS = int(os.environ.get("PAD_ONAP_MAX_WINDOWS", "60000"))


## Build feature windows

For LSTM forecasting, the notebook keeps segment lengths so sequences do not cross file boundaries.


In [ ]:
total_rows, label_counts = audit_csv_files(csv_files)
print("Total audited rows:", total_rows)
print("Mapped label counts:")
for cls in range(7):
    print(f"  {cls} {CLASS_NAMES[cls]:<15}: {label_counts.get(cls, 0):,}")

X, y, segments = build_windows(csv_files, MAX_WINDOWS)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("segments:", len(segments), "segment windows:", sum(segments))


## Temporal split and sequence construction

The split is performed inside each extracted segment, with a purge gap around the boundary to reduce leakage. Forecast labels are majority future class labels at +6w, +12w, +18w, and +24w.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

N_FEATURES = len(FEATURE_NAMES)
N_TIMESTEPS = int(os.environ.get("PAD_ONAP_N_TIMESTEPS", "12"))
HORIZON_STEPS = [6, 12, 18, 24]
HORIZON_LABELS = ["+6w", "+12w", "+18w", "+24w"]
LOOKAHEAD = int(os.environ.get("PAD_ONAP_LOOKAHEAD", "3"))
TRAIN_RATIO = float(os.environ.get("PAD_ONAP_TRAIN_RATIO", "0.8"))
PURGE_WINDOWS = int(os.environ.get("PAD_ONAP_PURGE_WINDOWS", "10"))

def split_by_segments(X, y, segments, train_ratio=TRAIN_RATIO, purge=PURGE_WINDOWS):
    train_X, train_y, train_segments = [], [], []
    test_X, test_y, test_segments = [], [], []
    start = 0
    for seg_len in segments:
        end = start + int(seg_len)
        Xs, ys = X[start:end], y[start:end]
        split = int(len(Xs) * train_ratio)
        left = max(0, split - purge)
        right = min(len(Xs), split + purge)
        if left > N_TIMESTEPS + max(HORIZON_STEPS) + LOOKAHEAD:
            train_X.append(Xs[:left])
            train_y.append(ys[:left])
            train_segments.append(left)
        if len(Xs) - right > N_TIMESTEPS + max(HORIZON_STEPS) + LOOKAHEAD:
            test_X.append(Xs[right:])
            test_y.append(ys[right:])
            test_segments.append(len(Xs) - right)
        start = end
    if not train_X or not test_X:
        raise RuntimeError("Temporal split produced empty train or test data. Increase MAX_WINDOWS or reduce horizons.")
    return (
        np.vstack(train_X), np.concatenate(train_y), train_segments,
        np.vstack(test_X), np.concatenate(test_y), test_segments,
    )

X_train_raw, y_train, seg_train, X_test_raw, y_test, seg_test = split_by_segments(X, y, segments)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test = scaler.transform(X_test_raw).astype(np.float32)

with open(MODELS_DIR / "stacked_lstm_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Train windows:", X_train.shape, "Test windows:", X_test.shape)
print("Train segments:", len(seg_train), "Test segments:", len(seg_test))

class ForecastDataset(Dataset):
    def __init__(self, X, y, segments, n_timesteps=N_TIMESTEPS, horizons=HORIZON_STEPS, lookahead=LOOKAHEAD):
        self.X_seq = []
        self.y_seq = []
        start = 0
        max_offset = max(horizons) + lookahead
        for seg_len in segments:
            end = start + int(seg_len)
            Xs = X[start:end]
            ys = y[start:end]
            upper = len(Xs) - n_timesteps - max_offset
            for i in range(max(0, upper)):
                self.X_seq.append(Xs[i:i + n_timesteps])
                labels = []
                for h in horizons:
                    fs = i + n_timesteps + h
                    fe = min(fs + lookahead, len(ys))
                    future = ys[fs:fe]
                    if len(future) == 0:
                        labels.append(0)
                    else:
                        vals, counts = np.unique(future, return_counts=True)
                        labels.append(int(vals[np.argmax(counts)]))
                self.y_seq.append(labels)
            start = end
        if self.X_seq:
            self.X_seq = np.asarray(self.X_seq, dtype=np.float32)
            self.y_seq = np.asarray(self.y_seq, dtype=np.int64)
        else:
            self.X_seq = np.empty((0, n_timesteps, X.shape[1]), dtype=np.float32)
            self.y_seq = np.empty((0, len(horizons)), dtype=np.int64)

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return torch.from_numpy(self.X_seq[idx]), torch.from_numpy(self.y_seq[idx])

train_ds = ForecastDataset(X_train, y_train, seg_train)
test_ds = ForecastDataset(X_test, y_test, seg_test)
print("Train sequences:", len(train_ds), "Test sequences:", len(test_ds))
if len(train_ds) == 0 or len(test_ds) == 0:
    raise RuntimeError("No forecast sequences produced. Adjust MAX_WINDOWS, N_TIMESTEPS, or HORIZON_STEPS.")


## Define and train stacked LSTM

This is a stacked LSTM only. It does not use a Transformer encoder.


In [ ]:
class StackedLSTMForecaster(nn.Module):
    def __init__(self, input_dim=N_FEATURES, hidden_dim=128, num_layers=3, dropout=0.3, n_horizons=4, n_classes=7):
        super().__init__()
        self.n_horizons = n_horizons
        self.n_classes = n_classes
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=False,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_horizons * n_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        logits = self.head(last)
        return logits.view(-1, self.n_horizons, self.n_classes)

BATCH_SIZE = int(os.environ.get("PAD_ONAP_LSTM_BATCH_SIZE", "256"))
EPOCHS = int(os.environ.get("PAD_ONAP_LSTM_EPOCHS", "40"))
LR = float(os.environ.get("PAD_ONAP_LSTM_LR", "0.001"))
PATIENCE = int(os.environ.get("PAD_ONAP_LSTM_PATIENCE", "8"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

model = StackedLSTMForecaster().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))

y0 = train_ds.y_seq[:, 0]
counts = np.bincount(y0, minlength=7).astype(np.float32)
weights = counts.sum() / np.maximum(counts, 1.0)
weights = weights / weights.mean()
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))

history = []
best_macro = -1.0
best_state = None
bad_epochs = 0

def evaluate(model, loader):
    model.eval()
    preds = [[] for _ in HORIZON_STEPS]
    labels = [[] for _ in HORIZON_STEPS]
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            pr = logits.argmax(dim=-1).cpu().numpy()
            y_np = yb.numpy()
            for h in range(len(HORIZON_STEPS)):
                preds[h].extend(pr[:, h].tolist())
                labels[h].extend(y_np[:, h].tolist())
    metrics = {}
    f1s = []
    accs = []
    for h, label in enumerate(HORIZON_LABELS):
        f1 = f1_score(labels[h], preds[h], average="macro", zero_division=0)
        acc = accuracy_score(labels[h], preds[h])
        metrics[f"macro_f1_{label}"] = float(f1)
        metrics[f"accuracy_{label}"] = float(acc)
        f1s.append(f1)
        accs.append(acc)
    metrics["macro_f1_mean"] = float(np.mean(f1s))
    metrics["accuracy_mean"] = float(np.mean(accs))
    return metrics, preds, labels

for epoch in range(EPOCHS):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = 0.0
        for h in range(len(HORIZON_STEPS)):
            loss = loss + criterion(logits[:, h, :], yb[:, h])
        loss = loss / len(HORIZON_STEPS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    scheduler.step()

    metrics, _, _ = evaluate(model, test_loader)
    row = {"epoch": epoch + 1, "loss": float(np.mean(losses)), **metrics}
    history.append(row)
    print(json.dumps(row, indent=2))

    if metrics["macro_f1_mean"] > best_macro:
        best_macro = metrics["macro_f1_mean"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

final_metrics, final_preds, final_labels = evaluate(model, test_loader)
print("Final metrics:")
print(json.dumps(final_metrics, indent=2))

torch.save(model.state_dict(), MODELS_DIR / "stacked_lstm_forecaster.pt")
pd.DataFrame(history).to_csv(TABLES_DIR / "stacked_lstm_training_history.csv", index=False)
with open(OUTPUT_DIR / "stacked_lstm_metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)
with open(OUTPUT_DIR / "stacked_lstm_metadata.json", "w") as f:
    json.dump({
        "feature_names": FEATURE_NAMES,
        "class_names": CLASS_NAMES,
        "n_timesteps": N_TIMESTEPS,
        "horizon_steps": HORIZON_STEPS,
        "horizon_labels": HORIZON_LABELS,
        "lookahead": LOOKAHEAD,
        "model": "stacked_lstm",
        "note": "Stacked LSTM only; no Transformer encoder; safe offline dataset processing."
    }, f, indent=2)


## Reports and plots

The saved metrics are evidence generated by this notebook run. Do not describe them as packet-level mitigation or ONAP/CNF results.


In [ ]:
import matplotlib.pyplot as plt

for h, label in enumerate(HORIZON_LABELS):
    labels_present = sorted(np.unique(np.concatenate([final_labels[h], final_preds[h]])))
    target_names = [CLASS_NAMES[i] for i in labels_present]
    report = classification_report(
        final_labels[h], final_preds[h],
        labels=labels_present, target_names=target_names,
        digits=4, zero_division=0
    )
    print("\n" + "=" * 72)
    print("Horizon", label)
    print(report)
    with open(OUTPUT_DIR / f"stacked_lstm_report_{label.replace('+', 'p')}.txt", "w") as f:
        f.write(report)
    cm = confusion_matrix(final_labels[h], final_preds[h], labels=labels_present)
    np.savetxt(TABLES_DIR / f"stacked_lstm_confusion_{label.replace('+', 'p')}.csv", cm, delimiter=",", fmt="%d")

hist = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist["epoch"], hist["macro_f1_mean"], marker="o", label="Mean macro F1")
ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1")
ax.set_ylim(0, 1.0)
ax.set_title("Stacked LSTM forecasting performance")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "stacked_lstm_macro_f1.png", dpi=160)
plt.show()

summary_df = pd.DataFrame([final_metrics]).T.reset_index()
summary_df.columns = ["metric", "value"]
summary_df.to_csv(TABLES_DIR / "stacked_lstm_metric_summary.csv", index=False)

summary = [
    "# PAD-ONAP stacked LSTM forecasting summary",
    "",
    "This notebook uses the ddos-train-v4 17-feature representation and a stacked LSTM forecaster.",
    "It predicts future traffic classes at +6w, +12w, +18w, and +24w.",
    "It does not use a Transformer encoder and does not generate packet traffic.",
    "",
    "## Final metrics",
    summary_df.to_markdown(index=False),
]
with open(OUTPUT_DIR / "stacked_lstm_summary.md", "w") as f:
    f.write("\n".join(summary))
print("Outputs:", OUTPUT_DIR)
